# RoadSign Assistant — Détection de panneaux routiers avec YOLO

## Objectif du projet

L’objectif de ce projet est de développer un modèle de vision par ordinateur capable de détecter plusieurs catégories de panneaux routiers dans une image.

Le modèle YOLO est entraîné sur un dataset exporté depuis Roboflow et évalué sur un jeu de test indépendant de l’entraînement.

**Technologies utilisées :**
- Python
- Ultralytics YOLO
- PyTorch
- Google Colab avec GPU
- Roboflow

**Dataset :** `traffic-sign-detection-yolov8-awuus`, version 11.

Le modèle entraîné est sauvegardé sous le nom `best.pt`. Les poids du modèle ne sont pas versionnés sur GitHub ; ils sont conservés séparément.


## 1. Classes détectées et dataset

Le modèle reconnaît 10 catégories de panneaux routiers :

1. `hump`
2. `no entry`
3. `no overtaking`
4. `no stopping`
5. `no u turn`
6. `parking`
7. `roadwork`
8. `roundabout`
9. `speed limit 40`
10. `stop`

Le dataset comprend :
- **Entraînement :** 2 352 images
- **Validation :** 219 images
- **Test :** 76 images
- **Total :** 2 647 images

Ces données permettent d’entraîner le modèle, de suivre ses performances pendant l’entraînement et d’évaluer ses résultats sur des images de test.


In [ ]:
import torch
import ultralytics
from pathlib import Path
import yaml

print("PyTorch :", torch.__version__)
print("Ultralytics :", ultralytics.__version__)
print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU utilisé :", torch.cuda.get_device_name(0))

## 2. Configuration du dataset

Le dataset a été téléchargé depuis Roboflow au format YOLOv8.

Dans cette exécution Google Colab, le dataset se trouve dans :

`/content/traffic-sign-detection-yolov8-11`

Le fichier `data.yaml` contient les chemins vers les images et la liste des classes.

Aucune clé API ou information secrète ne doit être inscrite directement dans le notebook.


In [ ]:
from pathlib import Path
import yaml

dataset_dir = Path("/content/traffic-sign-detection-yolov8-11")
data_yaml = dataset_dir / "data.yaml"

if not dataset_dir.exists():
    raise FileNotFoundError(
        "Dataset introuvable. Vérifiez qu'il est bien présent dans Colab."
    )

with open(data_yaml, "r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

data_config["train"] = str(dataset_dir / "train" / "images")
data_config["val"] = str(dataset_dir / "valid" / "images")
data_config["test"] = str(dataset_dir / "test" / "images")

with open(data_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False, allow_unicode=True)

print(yaml.safe_dump(data_config, sort_keys=False, allow_unicode=True))

## 3. Entraînement du modèle

Le modèle pré-entraîné `yolo26n.pt` est utilisé comme point de départ. L’entraînement est réalisé avec les paramètres suivants :

- Époques maximales : 30
- Taille des images : 640 × 640
- Batch size : 8
- Patience : 8
- Seed : 42

Les résultats et les poids sont enregistrés dans le dossier `roadsign_runs/traffic_signs`.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU non disponible. Activez le GPU dans les paramètres Colab."
    )

model = YOLO("yolo26n.pt")

train_results = model.train(
    data=str(data_yaml),
    epochs=30,
    imgsz=640,
    batch=8,
    patience=8,
    seed=42,
    project="/content/roadsign_runs",
    name="traffic_signs",
    plots=True
)

best_weights = Path(
    "/content/roadsign_runs/traffic_signs/weights/best.pt"
)

print("Modèle entraîné disponible :", best_weights.exists())

## 4. Évaluation sur le jeu de test

Les meilleurs poids enregistrés pendant l’entraînement sont utilisés pour évaluer le modèle sur le split `test`.

Les métriques principales sont :

- **Precision :** proportion des détections positives qui sont correctes.
- **Recall :** proportion des objets annotés qui sont détectés.
- **mAP50 :** moyenne de la précision moyenne avec un seuil IoU de 0,50.
- **mAP50-95 :** moyenne de la précision moyenne sur plusieurs seuils IoU.

Ces métriques permettent d’évaluer les performances de détection du modèle.


In [ ]:
from ultralytics import YOLO

best_weights = Path(
    "/content/roadsign_runs/traffic_signs/weights/best.pt"
)

model = YOLO(str(best_weights))

metrics = model.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    batch=8,
    plots=True
)

print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall : {metrics.box.mr:.4f}")
print(f"mAP50 : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")

## 5. Résultats obtenus

Lors de l’évaluation du modèle sur le jeu de test, les résultats obtenus étaient :

| Métrique | Résultat |
|---|---:|
| Precision | 90,33 % |
| Recall | 92,81 % |
| mAP50 | 94,16 % |
| mAP50-95 | 84,43 % |

### Interprétation

Le modèle obtient un mAP50 de 94,16 % sur le jeu de test utilisé. Les résultats sont encourageants pour ce projet de détection de panneaux routiers.

Cependant, le jeu de test contient seulement 76 images et 79 instances annotées. Ces résultats doivent donc être interprétés avec prudence. Une évaluation sur un jeu de test plus vaste et plus diversifié serait nécessaire pour mieux mesurer la capacité de généralisation du modèle.


## 6. Prédiction sur une image de test

Une image du jeu de test est sélectionnée afin de vérifier le comportement du modèle. La prédiction affiche les classes détectées et les boîtes englobantes correspondantes.

Le seuil de confiance utilisé pour cet exemple est de 0,25.


In [ ]:
from IPython.display import display, Image
from pathlib import Path

test_images_dir = dataset_dir / "test" / "images"

test_images = sorted(
    p for p in test_images_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

if not test_images:
    raise FileNotFoundError("Aucune image de test trouvée.")

image_path = test_images[0]

results = model.predict(
    source=str(image_path),
    conf=0.25,
    save=True
)

output_path = Path(str(results[0].save_dir)) / image_path.name

print("Image analysée :", image_path)
print("Image annotée :", output_path)

if output_path.exists():
    display(Image(filename=str(output_path)))

## 7. Conclusion et perspectives

Ce projet met en œuvre un pipeline de vision par ordinateur comprenant la préparation du dataset, l’entraînement d’un modèle YOLO, l’évaluation des performances et la visualisation de prédictions.

Les prochaines étapes consistent à intégrer le modèle dans une application Streamlit afin qu’un utilisateur puisse importer une image et visualiser les panneaux détectés.

### Limites

Le modèle a été évalué sur un jeu de test relativement petit. Des tests supplémentaires seraient nécessaires avant toute utilisation dans un contexte réel de sécurité routière.

Le modèle constitue un prototype de démonstration et ne remplace pas un système de reconnaissance routière certifié.
